In [1]:
import numpy as np

state = np.array([1,2,3,4])

print(state)

state[0:2] += state[2:4]

action_map = [(-1,1), (0, 1), (1, 1), (-1,0), (0, 0), (1, 0), (-1,-1), (0, -1), (1, -1)]

state[2:4] += action_map[1]

state

[1 2 3 4]


array([4, 6, 3, 5])

In [33]:
import jax
import jax.numpy as jnp
from flax import nnx

class StateModule(nnx.Module):
    def __init__(self):
        self.xy = nnx.Param(jnp.array(0))

@jax.jit
def update_jax_jit(state: StateModule):
    state.xy.value = jnp.array(2)
    return

@nnx.jit
def update_nnx_jit(state: StateModule):
    state.xy.value = jnp.array(3)
    return

state_module = StateModule()
print(f"State module starts as")
print(state_module)
update_jax_jit(state_module)
print(f"Update jax jit")
print(state_module)
update_nnx_jit(state_module)
print(f"Update nnx jit")
print(state_module)

@nnx.dataclass
class StateDataclass(nnx.Pytree):
    xy: jax.Array = nnx.data()

state_dataclass = StateDataclass(xy=jnp.array([1, 2]))
print(f"State dataclass starts as")
print(state_dataclass.xy)

@jax.jit
def update_dataclass_jax_jit(state: StateDataclass):
    state.xy = jnp.array([5, 6])
    return

@nnx.jit
def update_dataclass_nnx_jit(state: StateDataclass):
    state.xy = jnp.array([7, 8])
    return

@nnx.jit
def update_dataclass_nnx_jit_second(state: StateDataclass):
    def update(state: StateDataclass):
        state.xy = jnp.array([9, 10])
    update(state)
    return

update_dataclass_jax_jit(state_dataclass)
print(f"Update jax jit")
print(state_dataclass.xy)
update_dataclass_nnx_jit(state_dataclass)
print(f"Update nnx jit")
print(state_dataclass.xy)
update_dataclass_nnx_jit_second(state_dataclass)
print(f"Update nnx jit second")
print(state_dataclass.xy)

State module starts as
StateModule( # Param: 1 (4 B)
  xy=Param( # 1 (4 B)
    value=Array(0, dtype=int32, weak_type=True)
  )
)
Update jax jit
StateModule( # Param: 1 (4 B)
  xy=Param( # 1 (4 B)
    value=Array(0, dtype=int32, weak_type=True)
  )
)
Update nnx jit
StateModule( # Param: 1 (4 B)
  xy=Param( # 1 (4 B)
    value=Array(3, dtype=int32, weak_type=True)
  )
)
State dataclass starts as
[1 2]
Update jax jit
[1 2]
Update nnx jit
[7 8]
Update nnx jit second
[ 9 10]


In [34]:
@nnx.scan(in_axes=(nnx.Carry, 0), out_axes=(nnx.Carry, 0))
def loop(carry: StateModule, idx):
    carry.xy[...] = carry.xy.get_value() + 1
    return carry, carry.xy.get_value()

state_module_2 = StateModule()

loop_nnx_jit = nnx.jit(loop)

carry, accumulate = loop(state_module_2, jnp.arange(100))

print(accumulate)

[  1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17  18
  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35  36
  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53  54
  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71  72
  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89  90
  91  92  93  94  95  96  97  98  99 100]


In [35]:
@nnx.scan(in_axes=(nnx.Carry, 0), out_axes=(nnx.Carry, 0))
def loop(carry: StateDataclass, idx):
    carry.xy = carry.xy + 1
    return carry, carry.xy

state_dataclass_2 = StateDataclass(jnp.array(0))

loop_nnx_jit = nnx.jit(loop)

carry, accumulate = loop(state_dataclass_2, jnp.arange(100))

print(accumulate)

[  1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17  18
  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35  36
  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53  54
  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71  72
  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89  90
  91  92  93  94  95  96  97  98  99 100]


In [72]:
@nnx.jit
def cond_func(state: StateDataclass, rngs: nnx.Rngs):
    def increment(state: StateDataclass, rngs: nnx.Rngs):
        state.xy = state.xy + 2        
        return 
    
    def decrement(state: StateDataclass, rngs: nnx.Rngs):
        state.xy = state.xy - 2
        return
    
    nnx.cond(False, increment, decrement, state, rngs)

state_dataclass_3 = StateDataclass(jnp.array(0))

rngs = nnx.Rngs(0)

cond_func(state_dataclass_3, rngs)

print(state_dataclass_3)

StateDataclass(
  xy=Array(-2, dtype=int32, weak_type=True)
)
